# Mini projet : Assistant de sentiment avec réglage fin BERT

Bienvenue au mini-projet de la dernière journée !

Dans ce laboratoire, on va affiner (= "fine-tuner") `bert-base-uncased` sur des critiques de films,
évaluer le modèle obtenu, puis relier tout cela à un vrai scénario de support client : détecter
le sentiment (positif/négatif) d'un message client pour pouvoir escalader les clients mécontents
avant qu'ils ne quittent le service (le "churn").

**Scénario :** l'équipe d'analyse du support souhaite un signal de sentiment fiable sur des retours
longs, afin de pouvoir prioriser les clients en colère.

**Besoins environnementaux :** Python 3.9+, un runtime avec GPU (Colab, Kaggle, ou une machine GPU
locale) et environ 6 Go de VRAM libre. Le CPU seul fonctionne aussi, mais l'entraînement sera
beaucoup plus lent.

> **Note pour les débutants :** ce notebook contient un commentaire (précédé d'un `#`) au-dessus de
> presque chaque ligne de code, pour expliquer ce qu'elle fait et pourquoi. Les commentaires ne sont
> jamais exécutés par Python : ils sont uniquement là pour vous aider à comprendre.

## Installation des bibliothèques

On installe (ou met à jour) les bibliothèques nécessaires : `tensorflow` (le framework de deep
learning qu'on utilise ici), `tensorflow_datasets` (pour charger facilement le jeu de données
IMDB), `transformers` (la bibliothèque Hugging Face pour BERT), `accelerate` (utilitaires
d'entraînement) et `evaluate` (métriques prêtes à l'emploi). Ces packages ont déjà été utilisés
les jours précédents du cours.

In [ ]:
# Le "!" en début de ligne indique à Jupyter/Colab : "ce n'est pas du Python, exécute ceci
# comme une commande de terminal". pip est l'outil standard pour installer des paquets Python.
# -q = "quiet" (moins de texte affiché). On installe une seule fois, dans un environnement neuf.
!pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

## Importations et contrôle matériel

On commence toujours par confirmer les versions des bibliothèques et vérifier le matériel
disponible. Si vous voyez `GPU devices: []` (liste vide), cela veut dire qu'aucun GPU n'est détecté
: pensez à changer de runtime (`Runtime > Change runtime type > GPU` sur Google Colab).

In [ ]:
# "platform" est une bibliothèque standard de Python qui donne des informations sur
# l'environnement d'exécution (ici, on l'utilise juste pour afficher la version de Python).
import platform
# "tensorflow" (abrégé "tf" par convention universelle) est le framework de deep learning
# qu'on utilise dans ce notebook pour construire et entraîner le modèle.
import tensorflow as tf
# "tensorflow_datasets" (abrégé "tfds") permet de télécharger facilement des jeux de données
# publics tout prêts, comme IMDB Reviews qu'on va utiliser plus bas.
import tensorflow_datasets as tfds
# Depuis la bibliothèque "transformers" de Hugging Face, on importe :
# - BertTokenizer : pour découper du texte en "jetons" (tokens) compris par BERT
# - TFBertForSequenceClassification : le modèle BERT + une tête de classification,
#   en version TensorFlow (préfixe "TF")
from transformers import BertTokenizer, TFBertForSequenceClassification

# On affiche la version de Python utilisée.
print("Python version      :", platform.python_version())
# On affiche la version de TensorFlow installée.
print("TensorFlow version  :", tf.__version__)
# tf.config.list_physical_devices('GPU') renvoie la liste des GPU détectés par TensorFlow.
# Si la liste est vide ([]), aucun GPU n'est disponible et l'entraînement sera plus lent.
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

## Chargez le jeu de données IMDB Reviews

On utilise IMDB Reviews car il est équilibré (25 000 critiques positives / 25 000 négatives) et
déjà prêt à l'emploi via `tensorflow_datasets`.

In [ ]:
# tfds.load télécharge (ou réutilise une copie déjà téléchargée) le jeu de données "imdb_reviews".
# split=(...) demande à la fois la partie d'entraînement (TRAIN) et celle de test (TEST).
# as_supervised=True fait que chaque exemple est renvoyé comme une paire (texte, label),
# exactement ce que notre modèle attend (un texte en entrée, un label 0/1 à prédire).
# with_info=True renvoie aussi un objet "ds_info" qui décrit le jeu de données (nombre
# d'exemples, classes possibles, etc.).
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
# On affiche les informations du jeu de données (description, taille, etc.) pour vérifier
# que tout s'est bien chargé.
print(ds_info)

On regarde rapidement quelques exemples pour rendre le sentiment concret avant de continuer.

In [ ]:
# ds_train.take(2) prend seulement les 2 premiers exemples du jeu d'entraînement, pour un
# aperçu rapide sans tout parcourir (ce qui prendrait du temps inutilement ici).
for text, label in ds_train.take(2):
    # label est un tenseur contenant 0 (négatif) ou 1 (positif). label.numpy() extrait sa
    # valeur Python normale. On l'utilise dans une condition pour choisir le texte à afficher.
    print("Label:", "Positive" if label.numpy() else "Negative")
    # text.numpy() convertit le tenseur de texte en bytes Python, .decode() les transforme
    # en chaîne de caractères lisible. [:250] n'affiche que les 250 premiers caractères,
    # pour ne pas surcharger l'affichage avec une critique entière.
    print(text.numpy().decode()[:250], "...\n")

## Installation du Tokenizer & pipeline de données

BERT utilise la tokenisation **WordPiece** : les mots rares ou inconnus sont découpés en
sous-unités (sous-mots), ce qui garantit une bonne couverture du vocabulaire sans avoir besoin d'un
dictionnaire gigantesque. BERT ajoute aussi des jetons spéciaux : `[CLS]` (au début, utilisé pour
résumer la phrase pour la classification) et `[SEP]` (pour marquer la fin d'une phrase, ou séparer
deux phrases). Le masque d'attention indique au modèle quels jetons sont du vrai texte (1) et
lesquels ne sont que du remplissage ("padding", 0), pour que l'attention ne se calcule que sur les
entrées valides.

In [ ]:
# On fixe la longueur maximale (en nombre de jetons) de chaque critique après tokenisation.
# Les critiques plus longues seront coupées, les plus courtes complétées ("paddées") pour que
# toutes les critiques d'un même batch aient exactement la même longueur (nécessaire pour
# pouvoir les regrouper dans un même tenseur).
MAX_LENGTH = 256   # on coupe ou complète chaque critique à 256 jetons pour aligner les batches
# La taille de batch : le nombre d'exemples traités ensemble à chaque étape d'entraînement.
BATCH_SIZE = 16

# On charge le tokenizer officiel associé à "bert-base-uncased". "uncased" veut dire que le
# tokenizer ignore les majuscules/minuscules (tout est mis en minuscules avant tokenisation).
# do_lower_case=True confirme explicitement ce comportement.
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
# .name_or_path indique d'où vient ce tokenizer (utile pour vérifier qu'on a bien chargé
# le bon modèle de référence).
print("Tokenizer loaded:", tokenizer.name_or_path)

On importe ce tokenizer pour réutiliser le même vocabulaire que celui du modèle BERT pré-entraîné
(sinon les jetons ne correspondraient pas aux poids appris par le modèle).

On convertit maintenant les textes bruts en identifiants de jetons (`input_ids`), masques
d'attention (`attention_mask`) et identifiants de segments (`token_type_ids`).

In [ ]:
# On définit une fonction qui prend UN texte (review_input) et le tokenise avec BERT.
# Le texte peut arriver sous plusieurs formes différentes selon le contexte d'appel
# (bytes, tenseur TensorFlow, ou déjà une chaîne), donc on gère les 3 cas possibles.
def encode_review(review_input):
    # Si c'est des "bytes" bruts (une suite d'octets), on les décode en texte UTF-8.
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    # Si c'est un tenseur TensorFlow (il possède une méthode .numpy()), on en extrait
    # la valeur numpy, puis on décode ces bytes en texte.
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    # Sinon (cas improbable mais on se protège), on force juste la conversion en chaîne.
    else:
        review_text = str(review_input)

    # tokenizer.encode_plus fait tout le travail de tokenisation BERT en une seule fonction :
    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,      # ajoute automatiquement les jetons [CLS] et [SEP]
        max_length=MAX_LENGTH,         # longueur cible fixée plus haut (256)
        padding="max_length",          # complète les textes courts jusqu'à MAX_LENGTH
        truncation=True,                # coupe les textes trop longs à MAX_LENGTH
        return_attention_mask=True,     # demande explicitement le masque d'attention en retour
        return_token_type_ids=True,     # demande aussi les identifiants de segment (tous à 0 ici,
                                         # car on n'a qu'une seule phrase, pas une paire de phrases)
    )

# tf.data (le pipeline de données de TensorFlow) ne peut pas appeler directement une fonction
# Python "normale" comme encode_review à l'intérieur de .map() -- il a besoin d'une fonction
# "compatible TensorFlow". On définit donc une fonction intermédiaire tf_encode qui utilise
# tf.py_function pour faire le pont entre TensorFlow et notre code Python/Hugging Face.
def tf_encode(text, label):
    # tf.py_function exécute une fonction Python normale (ici une fonction "lambda" anonyme)
    # à l'intérieur du graphe TensorFlow. func=... est la fonction à exécuter, inp=[text] est
    # la liste des arguments à lui passer, et Tout=[...] précise le type de chaque valeur
    # renvoyée (ici 3 tableaux d'entiers 32 bits : input_ids, attention_mask, token_type_ids).
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    # On retourne un dictionnaire nommé (ce que le modèle BERT attend en entrée) associé
    # à son label inchangé. encoded[0], [1], [2] correspondent dans l'ordre à input_ids,
    # attention_mask, token_type_ids (l'ordre dans lequel encode_plus les a renvoyés).
    return {
        "input_ids": encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label

# On définit une fonction qui prend un jeu de données brut (textes, labels) et le transforme
# en un pipeline prêt pour l'entraînement : tokenisé, mélangé, regroupé en lots, préchargé.
def prepare_dataset(dataset):
    return (
        dataset
        # .map applique tf_encode à chaque exemple. num_parallel_calls=AUTOTUNE laisse
        # TensorFlow choisir automatiquement le bon niveau de parallélisme pour aller plus vite.
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        # .shuffle(2000) mélange les exemples (avec un buffer de 2000 exemples à la fois),
        # pour que le modèle ne voie pas toujours les données dans le même ordre.
        .shuffle(2000)
        # .batch(BATCH_SIZE) regroupe les exemples par lots de 16 (valeur définie plus haut).
        .batch(BATCH_SIZE)
        # .prefetch(AUTOTUNE) prépare déjà le batch suivant pendant que le modèle traite
        # le batch actuel, ce qui accélère l'entraînement en évitant les temps d'attente.
        .prefetch(tf.data.AUTOTUNE)
    )

# On applique cette préparation aux jeux d'entraînement et de test chargés plus haut.
train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

`tf.py_function` permet de continuer à utiliser la logique de tokenisation Hugging Face à
l'intérieur d'un pipeline `tf.data`, sans avoir à jongler manuellement avec des tableaux NumPy. Le
mélange (`shuffle`) et le préchargement (`prefetch`) stabilisent aussi le débit d'entraînement.

## Initialiser le modèle de réglage fin

On charge `TFBertForSequenceClassification`, qui regroupe déjà l'encodeur BERT pré-entraîné et une
tête de classification (ajoutée automatiquement, initialisée aléatoirement) qui produira 2 scores
en sortie — un pour "négatif", un pour "positif".

In [ ]:
# TFBertForSequenceClassification.from_pretrained charge les poids pré-entraînés de BERT
# (appris sur énormément de texte) ET ajoute une nouvelle couche de classification par-dessus.
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,           # 2 classes en sortie : négatif (0) et positif (1)
    use_safetensors=False,  # on force le format de poids "classique" (.bin) plutôt que safetensors
)

# tf.keras.optimizers.Adam est l'algorithme qui ajustera les poids du modèle pendant
# l'entraînement, pour réduire l'erreur petit à petit.
# learning_rate=2e-5 (0.00002) est une valeur volontairement très petite, car on ne fait que
# du "fine-tuning" (ajustement léger) d'un modèle déjà très performant, pas un entraînement
# depuis zéro -- un taux trop élevé risquerait de "casser" les connaissances déjà apprises.
# epsilon=1e-8 est une petite constante numérique qui stabilise les calculs internes d'Adam.
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
# La fonction de perte mesure l'écart entre la prédiction du modèle et la vraie étiquette.
# SparseCategoricalCrossentropy convient quand les labels sont de simples entiers (0 ou 1),
# par opposition à des vecteurs "one-hot" comme [1, 0] ou [0, 1].
# from_logits=True indique que le modèle renvoie des scores bruts (logits), pas encore
# des probabilités -- la fonction de perte s'occupe elle-même d'appliquer le softmax.
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
# On choisit la métrique à suivre pendant l'entraînement : ici, la précision (accuracy),
# c'est-à-dire la proportion de prédictions exactement correctes.
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

# model.compile assemble le modèle avec son optimiseur, sa fonction de perte et ses métriques :
# c'est cette étape qui "prépare" le modèle à être entraîné avec model.fit() juste après.
model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
# model.summary() affiche un résumé de l'architecture du modèle (nombre de couches,
# nombre total de paramètres, etc.) : pratique pour vérifier que tout est bien chargé.
model.summary()

On réutilise ainsi les ~110 millions de paramètres déjà appris par BERT sur BooksCorpus +
Wikipédia. On ne fait du fine-tuning que pour quelques époques, c'est pourquoi un taux
d'apprentissage de l'ordre de `2e-5` est une valeur standard pour ce genre de tâche.

## Entraîner et surveiller

Sur un GPU T4 (celui fourni gratuitement par Google Colab), deux époques prennent environ 15
minutes. Surveillez à la fois la précision d'entraînement et de validation pour repérer un éventuel
sur-apprentissage (la précision d'entraînement qui continue de monter alors que celle de validation
stagne ou redescend).

In [ ]:
# Le nombre d'époques : combien de fois le modèle va voir l'intégralité du jeu d'entraînement.
EPOCHS = 2  # on peut passer à 3 si le temps disponible le permet

# model.fit() lance réellement l'entraînement. C'est la méthode Keras standard pour entraîner
# un modèle : elle gère automatiquement la boucle (lire un batch, calculer la perte, ajuster
# les poids, répéter), pendant le nombre d'époques demandé.
history = model.fit(
    x=train_ds,               # le pipeline de données d'entraînement préparé plus haut
    validation_data=test_ds,  # on évalue sur le jeu de test après chaque époque, pour suivre
                               # la progression sur des données que le modèle n'a jamais vues
    epochs=EPOCHS,            # nombre d'époques défini ci-dessus (2)
)

Observez comment le plateau de précision de validation indique quand il devient inutile de
continuer l'entraînement (les gains supplémentaires deviennent marginaux, voire négatifs si le
modèle commence à sur-apprendre). Pensez aussi à garder une capture d'écran des courbes
d'apprentissage (disponibles dans `history.history`) pour votre portfolio.

## Évaluer sur le jeu de test suspendu

Même si les métriques de validation sont déjà rapportées pendant `model.fit`, on refait une
évaluation explicite sur le pipeline de test, pour imiter une vérification qualité de type
"production" (séparée de la boucle d'entraînement elle-même).

In [ ]:
# model.evaluate() fait passer tout le jeu de test dans le modèle et calcule la perte +
# les métriques (ici, l'accuracy), sans faire aucun ajustement des poids (contrairement à .fit()).
eval_metrics = model.evaluate(test_ds)
# eval_metrics est une liste : [perte, accuracy] (dans l'ordre où on les a définis dans .compile()).
# On affiche les deux valeurs avec 4 décimales pour une lecture précise.
print(f"Test Loss: {eval_metrics[0]:.4f}, Test Accuracy: {eval_metrics[1]:.4f}")

Ici, vérifiez si l'accuracy dépasse le seuil de référence d'environ 0.90 utilisé en classe.
C'est aussi l'occasion de discuter avec votre équipe des taux d'erreur acceptables pour une vraie
équipe de support : une erreur sur 10 messages peut avoir des conséquences très différentes selon
qu'on parle d'un simple routage interne ou d'une décision automatique visible du client.

## Construire un assistant d'inférence réutilisable

On regroupe toute la logique de prédiction dans une seule fonction, pour pouvoir y copier-coller
de vraies transcriptions de support et obtenir un score instantané.

In [ ]:
# On définit une fonction qui prend un texte brut et renvoie le sentiment prédit + la confiance.
def predict_sentiment(text: str):
    # On encode le texte d'entrée, exactement comme pour l'entraînement, mais pour un seul texte.
    encoded_input = tokenizer.encode_plus(
        text,
        add_special_tokens=True,    # ajoute [CLS] et [SEP] comme pendant l'entraînement
        max_length=MAX_LENGTH,       # même longueur cible que pendant l'entraînement (256)
        padding='max_length',        # complète jusqu'à MAX_LENGTH si le texte est plus court
        truncation=True,              # coupe si le texte est plus long que MAX_LENGTH
        return_attention_mask=True,   # on a besoin du masque d'attention pour le modèle
        return_token_type_ids=True,   # idem pour les identifiants de segment
        return_tensors='tf'           # on demande directement des tenseurs TensorFlow en retour
                                       # (plus pratique ici, car on va les donner au modèle tel quel)
    )

    # On prépare l'entrée sous la forme exacte d'un dictionnaire attendu par le modèle BERT.
    inputs = {
        'input_ids': encoded_input['input_ids'],
        'attention_mask': encoded_input['attention_mask'],
        'token_type_ids': encoded_input['token_type_ids']
    }

    # On appelle le modèle comme une fonction : model(inputs) renvoie un objet contenant
    # plusieurs informations ; [0] sélectionne les logits (scores bruts avant transformation
    # en probabilités), qui sont le premier élément renvoyé.
    logits = model(inputs)[0]

    # tf.nn.softmax transforme les logits bruts en probabilités qui se somment à 1.
    # axis=-1 applique le softmax sur le dernier axe (celui des 2 classes).
    # [0] enlève la dimension de batch (on n'a qu'un seul exemple ici).
    probs = tf.nn.softmax(logits, axis=-1)[0]

    # tf.argmax(probs) trouve l'indice de la probabilité la plus élevée : 0 (négatif) ou
    # 1 (positif). .numpy() convertit ce résultat en simple entier Python.
    predicted_class = tf.argmax(probs).numpy()
    # On convertit ce numéro de classe en texte lisible.
    label = "Positive" if predicted_class == 1 else "Negative"

    # On retourne le label prédit, ainsi que la probabilité (confiance) associée à cette classe.
    # float(...) convertit la valeur tensorielle en nombre Python normal.
    return label, float(probs[predicted_class])


# On teste notre fonction sur un exemple inspiré d'un vrai message de support client.
custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
# :.3f affiche la confiance avec 3 décimales, pour une lecture précise du score.
print(f"Prediction: {label} (confidence={confidence:.3f})")

Pour l'exemple ci-dessus (un message mitigé : un problème au départ, mais résolu poliment), on
s'attend à une prédiction `Positive` avec une confiance assez proche de `0.5` (légèrement plus ou
moins) — le message contient à la fois des éléments négatifs et positifs, donc le modèle n'est pas
totalement tranché.

**Pourquoi la confiance compte :** ces scores sont essentiels pour décider si on peut répondre
automatiquement à un client, ou s'il faut au contraire escalader la conversation vers un agent
humain (par exemple, en dessous d'un certain seuil de confiance).

## Réflexion et prochaines étapes

- **Pourquoi l'ajustement fin est important :** vous avez réutilisé un point de contrôle public
  (BERT pré-entraîné) pour atteindre plus de 90 % de précision avec un minimum de données et de
  temps d'entraînement.
- **Compétences transférables :** cette même approche s'applique aussi à des tâches de
  classification en RH, juridique, ou analyse produit.
- **Pour aller plus loin :** adaptation de domaine (entraîner sur les emails de votre propre
  entreprise), points de contrôle multilingues (DistilBERT multilingue, XLM-R), et surveillance en
  production (détection de dérive des données, tableaux de bord de suivi).

Répondez aux questions de réflexion suivantes directement dans les cellules markdown ci-dessous :

1. **Quel levier (nettoyage des données, hyperparamètres, plus d'époques) a le plus amélioré les
   résultats ?**

   _Réponse :_ (à remplir par l'apprenant)

2. **Où ajouteriez-vous des garde-fous avant de déployer ce signal de sentiment en direct ?**

   _Réponse :_ (à remplir par l'apprenant)

3. **Quelles parties prenantes en bénéficient le plus (responsable support, chef de produit,
   responsable conformité) ?**

   _Réponse :_ (à remplir par l'apprenant)